# Exploration de Category Translation — phase 1

Lecture seule sur C:, .venv du projet. Exécuter de haut en bas depuis la racine ou notebooks/.
Grain attendu : une correspondance catégorie portugaise → libellé anglais ; cardinalités à vérifier.
Aucune traduction ajoutée et aucun validateur appelé. Rapport : docs/CATEGORY_TRANSLATION_EXPLORATION.md.
Code exécuté avec Python le 2026-10-24 ; notebook livré sans sorties sauvegardées ni lancement de noyau Jupyter.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "AGENTS.md").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "AGENTS.md").is_file()
paths = {
    "translation": PROJECT_ROOT / "data" / "product_category_name_translation.csv",
    "products": PROJECT_ROOT / "data" / "olist_products_dataset.csv",
}
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1048576), b""):
            digest.update(block)
    return digest.hexdigest()
hashes_before = {name:sha256(path) for name,path in paths.items()}
translation = pd.read_csv(paths["translation"])
raw = pd.read_csv(paths["translation"], dtype="string", keep_default_na=False)
products = pd.read_csv(paths["products"], usecols=["product_id", "product_category_name"])
PT = "product_category_name"
EN = "product_category_name_english"

def records(frame):
    return json.loads(frame.to_json(orient="records", force_ascii=True))
def counts(series):
    return {str(key):int(value) for key,value in series.value_counts().items()}

report = {
    "environment": {"python":platform.python_version(), "pandas":pd.__version__,
                    "executable":sys.executable},
    "source_sha256": hashes_before,
    "source_size_bytes": paths["translation"].stat().st_size,
    "shape":list(translation.shape),
    "dtypes": {col:str(dtype) for col,dtype in translation.dtypes.items()},
    "missing": {col:int(value) for col,value in translation.isna().sum().items()},
    "empty": {col:int(raw[col].eq("").sum()) for col in raw},
    "whitespace_only_nonempty": {col:int((raw[col].ne("") & raw[col].str.strip().eq("")).sum()) for col in raw},
    "exact_duplicate_excess":int(raw.duplicated().sum()),
    "exact_duplicate_involved_rows":int(raw.duplicated(keep=False).sum()),
}
report

## Clés, contradictions et variations d'écriture

Le doublon d'une catégorie portugaise et une traduction anglaise partagée sont deux situations différentes.
Une contradiction signifie plusieurs libellés anglais distincts non vides pour la même catégorie portugaise exacte.
La normalisation exploratoire sert uniquement à détecter des collisions ; elle ne corrige aucune valeur.
Un motif inhabituel n'est pas automatiquement une erreur de traduction.

In [ ]:
report["identifiers"] = {}
report["writing"] = {}
for col in [PT, EN]:
    text = raw[col]
    key = (text.str.casefold().str.normalize("NFKD")
           .str.replace(r"[\u0300-\u036f]", "", regex=True)
           .str.replace(r"[^a-z0-9]+", "_", regex=True)
           .str.strip("_"))
    frequency = text.value_counts()
    variants = pd.DataFrame({"raw":text, "key":key}).groupby("key").raw.nunique()
    report["identifiers"][col] = {
        "distinct":int(text.nunique()),
        "repeated_values":int(frequency.gt(1).sum()),
        "duplicate_involved_rows":int(text.duplicated(keep=False).sum()),
        "duplicate_excess":int(text.duplicated().sum()),
    }
    unusual = ~text.str.fullmatch("[a-z0-9]+(?:_[a-z0-9]+)*", na=False)
    report["writing"][col] = {
        "outer_whitespace_rows":int(text.ne(text.str.strip()).sum()),
        "contains_whitespace_rows":int(text.str.contains(r"\s", regex=True, na=False).sum()),
        "non_ascii_rows":int(text.str.contains(r"[^\x00-\x7f]", regex=True, na=False).sum()),
        "non_lowercase_rows":int(text.ne(text.str.lower()).sum()),
        "unusual_slug_rows":int(unusual.sum()),
        "unusual_values":text[unusual].tolist(),
        "normalized_collision_groups":int(variants.gt(1).sum()),
        "values_with_digits":text[text.str.contains("[0-9]", regex=True, na=False)].tolist(),
    }
usable = raw[raw[PT].str.strip().ne("") & raw[EN].str.strip().ne("")]
english_counts = usable.groupby(PT)[EN].nunique()
portuguese_counts = usable.groupby(EN)[PT].nunique()
report["mapping"] = {
    "portuguese_categories_with_conflicting_translations":int(english_counts.gt(1).sum()),
    "conflicting_rows":records(usable[usable[PT].isin(english_counts[english_counts.gt(1)].index)]),
    "english_labels_shared_by_multiple_portuguese_categories":int(portuguese_counts.gt(1).sum()),
    "shared_english_rows":records(usable[usable[EN].isin(portuguese_counts[portuguese_counts.gt(1)].index)]),
}
report["identifiers"], report["writing"], report["mapping"]

## Couverture Products

Séparer une catégorie produit absente d'une catégorie renseignée sans traduction.
Contrôler la jointure many_to_one avant de compter les produits : plusieurs lignes de traduction pour une catégorie pourraient multiplier les lignes.
Une traduction partagée par plusieurs catégories n'est pas une ambiguïté dans le sens portugais → anglais.

In [ ]:
missing_product_category = (
    products[PT].isna() | products[PT].astype("string").str.strip().eq("").fillna(False))
untranslated = ~missing_product_category & ~products[PT].isin(usable[PT])
unused = usable.loc[~usable[PT].isin(products.loc[~missing_product_category, PT])]
ambiguous_categories = english_counts[english_counts.gt(1)].index
report["products_relation"] = {
    "product_rows":len(products),
    "distinct_present_categories":int(products.loc[~missing_product_category, PT].nunique()),
    "missing_category_product_rows":int(missing_product_category.sum()),
    "untranslated_categories":counts(products.loc[untranslated, PT]),
    "untranslated_distinct_categories":int(products.loc[untranslated, PT].nunique()),
    "untranslated_product_rows":int(untranslated.sum()),
    "translated_product_rows":int((~missing_product_category & ~untranslated).sum()),
    "unused_translated_distinct_categories":int(unused[PT].nunique()),
    "unused_translated_rows":records(unused),
    "product_rows_on_ambiguous_categories":int(products[PT].isin(ambiguous_categories).sum()),
    "target_categories": {
        category: {"product_rows":int(products[PT].eq(category).sum()),
                   "translation_rows":int(raw[PT].eq(category).sum())}
        for category in ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"]
    },
}
if not raw[PT].duplicated().any():
    joined = products.merge(raw, on=PT, how="left", validate="many_to_one", indicator=True)
    assert len(joined) == len(products)
    report["products_relation"]["joined_rows"] = len(joined)
    report["products_relation"]["unmatched_rows_including_missing_category"] = int(joined._merge.eq("left_only").sum())
else:
    report["products_relation"]["join_blocked_by_duplicate_portuguese_category"] = True
assert hashes_before == {name:sha256(path) for name,path in paths.items()}
report["products_relation"]

## Contrat proposé — en attente de validation

Deux colonnes obligatoires, catégorie portugaise non vide et unique, traduction anglaise non vide ; contradictions explicites si une même catégorie possède plusieurs traductions distinctes.
Ne pas imposer l'unicité anglaise : une convergence de libellés peut être légitime.
Commentaires séparés sur les unités : doublon de catégorie = lignes impliquées ; contradiction = catégories et lignes concernées. Ces motifs peuvent se chevaucher.
Indicateurs : catégories Products non traduites, traductions inutilisées, anglais partagé et variations d'écriture.
Ne pas ajouter les deux traductions absentes ni transformer l'absence de catégorie Products en erreur de la table de traduction.
Aucune politique de nettoyage/dédoublonnage automatique. La validité linguistique d'une traduction ne peut pas être démontrée par ces contrôles structurels.

In [ ]:
print(json.dumps(report, ensure_ascii=True, indent=2))